# 1일차 — Tabular-based Methods

2026-07-27 (월) · 강화학습의 뼈대가 되는 MDP·동적계획법·시간차 학습을 표 기반 방법으로 익힙니다.

> 위에서부터 순서대로 실행하세요. 뒤 교시가 앞 교시의 변수·클래스를 그대로 이어 씁니다.


## 1교시 · 강화학습 소개

`09:30 ~ 10:30` · `bandit_epsilon_greedy.py`

- 강화학습이 지도학습·비지도학습과 어떻게 다른지 설명할 수 있다
- 에이전트-환경 상호작용 루프(상태·행동·보상)를 이해한다
- 탐험(Exploration)과 활용(Exploitation)의 트레이드오프를 이해한다


In [ ]:
import numpy as np

# 10개의 슬롯머신(Multi-Armed Bandit)으로 보는 탐험 vs 활용
np.random.seed(0)
n_arms = 10
true_means = np.random.normal(0, 1, n_arms)   # 각 팔의 실제 평균 보상

def run_bandit(epsilon, steps=2000):
    Q = np.zeros(n_arms)        # 행동가치 추정치
    N = np.zeros(n_arms)        # 각 팔을 당긴 횟수
    rewards = []
    for t in range(steps):
        if np.random.rand() < epsilon:
            a = np.random.randint(n_arms)      # 탐험
        else:
            a = np.argmax(Q)                   # 활용
        r = np.random.normal(true_means[a], 1) # 보상 샘플
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]              # 증분 평균 업데이트
        rewards.append(r)
    return np.mean(rewards)

for eps in [0.0, 0.01, 0.1, 0.5]:
    print(f"epsilon={eps:4.2f}  평균 보상 = {run_bandit(eps):.3f}")

# epsilon=0(탐험 없음)은 나쁜 팔에 갇히고,
# 0.5(과도한 탐험)는 보상을 낭비합니다. 0.1 근처가 균형점.

## 2교시 · MDP 소개

`10:30 ~ 11:30` · `gridworld_mdp.py`

- 마르코프 결정 과정(MDP)의 5요소 (S, A, P, R, γ)를 설명할 수 있다
- 상태가치함수 V(s)와 행동가치함수 Q(s,a)의 차이를 이해한다
- 벨만 방정식의 재귀 구조를 이해한다


In [ ]:
import numpy as np

# 4x4 GridWorld MDP 정의 — 이후 세션에서 계속 재사용합니다
# 상태: 0~15 (왼쪽 위에서 오른쪽 아래로), 0과 15는 종료 상태
# 행동: 0=상, 1=하, 2=좌, 3=우 / 보상: 이동마다 -1

N = 4
n_states = N * N
n_actions = 4
TERMINALS = [0, n_states - 1]

def step(s, a):
    """결정적 전이: (다음상태, 보상) 반환"""
    if s in TERMINALS:
        return s, 0
    r, c = divmod(s, N)
    if a == 0: r = max(r - 1, 0)
    elif a == 1: r = min(r + 1, N - 1)
    elif a == 2: c = max(c - 1, 0)
    elif a == 3: c = min(c + 1, N - 1)
    return r * N + c, -1

# 전이 텐서 P[s][a] = (s', r) 를 미리 만들어 두면 DP가 간단해집니다
P = [[step(s, a) for a in range(n_actions)] for s in range(n_states)]

# 무작위 정책의 한 에피소드 시뮬레이션
s, trajectory = 5, []
rng = np.random.default_rng(42)
while s not in TERMINALS:
    a = rng.integers(n_actions)
    s_next, r = P[s][a]
    trajectory.append((s, a, r))
    s = s_next
print(f"에피소드 길이: {len(trajectory)}, 총 보상: {sum(t[2] for t in trajectory)}")

## 3교시 · Dynamic Programming 소개

`11:30 ~ 12:30` · `policy_evaluation.py`

- 정책 평가(Policy Evaluation)와 정책 개선(Policy Improvement)을 구분한다
- 정책 반복(PI)과 가치 반복(VI)의 차이를 설명할 수 있다
- DP가 model-based 방법인 이유와 한계를 이해한다


In [ ]:
import numpy as np

# 무작위 정책에 대한 반복적 정책 평가 (4x4 GridWorld, 앞 세션의 P 재사용)
gamma = 1.0
theta = 1e-6          # 수렴 판정 기준

V = np.zeros(n_states)
iteration = 0
while True:
    delta = 0.0
    V_new = V.copy()
    for s in range(n_states):
        if s in TERMINALS:
            continue
        # 무작위 정책: 4방향 각 0.25 확률
        v = 0.0
        for a in range(n_actions):
            s_next, r = P[s][a]
            v += 0.25 * (r + gamma * V[s_next])
        V_new[s] = v
        delta = max(delta, abs(v - V[s]))
    V = V_new
    iteration += 1
    if delta < theta:
        break

print(f"{iteration}회 반복 후 수렴")
print(np.round(V.reshape(4, 4), 1))
# 종료 상태에서 멀수록 가치가 낮아지는(-22 근처) 것을 확인하세요

## 4교시 · Policy Iteration, Value Iteration 구현

`13:30 ~ 14:30` · `pi_vi_gridworld.py`

- 4x4 GridWorld에서 정책 반복을 NumPy로 구현한다
- 가치 반복을 구현하고 두 방법의 수렴 결과를 비교한다


In [ ]:
import numpy as np

gamma = 1.0

def q_from_v(V, s):
    """상태 s에서 각 행동의 Q값 계산"""
    return np.array([P[s][a][1] + gamma * V[P[s][a][0]]
                     for a in range(n_actions)])

# ── 정책 반복 (Policy Iteration) ──────────────────
def policy_iteration():
    policy = np.zeros(n_states, dtype=int)      # 모든 상태에서 행동 0
    while True:
        # 1) 정책 평가
        # 평가 sweep에 상한을 둡니다 (MAX_SWEEP).
        # 이유: 초기 정책(모두 '상')은 맨 윗줄에서 벽에 막혀 제자리에 머뭅니다.
        # 그러면 V[s] = -1 + 1.0 * V[s] 라서 gamma=1일 때 값이 발산해
        # delta < 1e-6 조건이 영원히 성립하지 않습니다 (무한 루프).
        # 상한을 두고 개선 단계로 넘어가면 다음 정책은 종료 상태에 도달하므로
        # 이후로는 정상 수렴합니다 — 이를 modified policy iteration이라 부릅니다.
        MAX_SWEEP = 1000
        V = np.zeros(n_states)
        for _ in range(MAX_SWEEP):
            delta = 0.0
            for s in range(n_states):
                if s in TERMINALS: continue
                s_next, r = P[s][policy[s]]
                v = r + gamma * V[s_next]
                delta = max(delta, abs(v - V[s]))
                V[s] = v
            if delta < 1e-6: break
        # 2) 정책 개선
        stable = True
        for s in range(n_states):
            if s in TERMINALS: continue
            best_a = np.argmax(q_from_v(V, s))
            if best_a != policy[s]:
                stable = False
                policy[s] = best_a
        if stable:
            return policy, V

# ── 가치 반복 (Value Iteration) ───────────────────
def value_iteration():
    V = np.zeros(n_states)
    while True:
        delta = 0.0
        for s in range(n_states):
            if s in TERMINALS: continue
            v = q_from_v(V, s).max()            # max가 곧 개선
            delta = max(delta, abs(v - V[s]))
            V[s] = v
        if delta < 1e-6: break
    policy = np.array([np.argmax(q_from_v(V, s)) for s in range(n_states)])
    return policy, V

arrows = np.array(['↑', '↓', '←', '→'])
pi_policy, pi_V = policy_iteration()
vi_policy, vi_V = value_iteration()
print("PI 최적 정책:"); print(arrows[pi_policy].reshape(4, 4))
print("VI 최적 정책:"); print(arrows[vi_policy].reshape(4, 4))
print("두 가치함수 일치:", np.allclose(pi_V, vi_V))

## 5교시 · Monte-Carlo 방법, Temporal Difference 방법 소개

`14:30 ~ 15:30` · `mc_vs_td_prediction.py`

- 모델 없이(model-free) 가치를 추정하는 두 접근을 이해한다
- MC의 낮은 편향·높은 분산, TD의 부트스트래핑·낮은 분산 특성을 비교한다
- TD(0) 업데이트 식을 쓸 수 있다


In [ ]:
import numpy as np

# 무작위 정책의 V를 MC와 TD(0)로 각각 추정해 비교 (4x4 GridWorld)
rng = np.random.default_rng(0)
alpha, gamma = 0.05, 1.0

def gen_episode():
    s = rng.integers(1, n_states - 1)
    episode = []
    while s not in TERMINALS:
        a = rng.integers(n_actions)
        s_next, r = P[s][a]
        episode.append((s, r))
        s = s_next
    return episode

# ── First-visit Monte-Carlo ──
V_mc = np.zeros(n_states)
for _ in range(5000):
    episode = gen_episode()
    G, visited = 0.0, set()
    for s, r in reversed(episode):     # 뒤에서부터 리턴 누적
        G = r + gamma * G
        if s not in visited:           # first-visit
            visited.add(s)
            V_mc[s] += alpha * (G - V_mc[s])

# ── TD(0) ──
V_td = np.zeros(n_states)
for _ in range(5000):
    s = rng.integers(1, n_states - 1)
    while s not in TERMINALS:
        a = rng.integers(n_actions)
        s_next, r = P[s][a]
        td_error = r + gamma * V_td[s_next] - V_td[s]
        V_td[s] += alpha * td_error
        s = s_next

print("MC 추정:"); print(np.round(V_mc.reshape(4, 4), 1))
print("TD 추정:"); print(np.round(V_td.reshape(4, 4), 1))
# 둘 다 DP 정답(-14, -20, -22...)에 근접하는지 확인하세요

## 6교시 · SARSA와 Q-Learning 소개

`15:30 ~ 16:30` · `update_rules.py`

- TD 제어에서 SARSA와 Q-Learning의 업데이트 식을 구분한다
- On-policy와 Off-policy의 차이를 설명할 수 있다


In [ ]:
# 두 알고리즘의 차이는 단 한 줄 — TD 목표(target)의 정의

# SARSA (on-policy): 다음 "실제" 행동 a'의 Q값 사용
def sarsa_update(Q, s, a, r, s_next, a_next, alpha=0.1, gamma=0.99):
    target = r + gamma * Q[s_next][a_next]        # 실제 선택한 a'
    Q[s][a] += alpha * (target - Q[s][a])

# Q-Learning (off-policy): 다음 상태의 "최대" Q값 사용
def q_learning_update(Q, s, a, r, s_next, alpha=0.1, gamma=0.99):
    target = r + gamma * max(Q[s_next])           # max — 실제 행동과 무관
    Q[s][a] += alpha * (target - Q[s][a])

# 행동 선택은 둘 다 epsilon-greedy
import random
def epsilon_greedy(Q, s, n_actions, epsilon=0.1):
    if random.random() < epsilon:
        return random.randrange(n_actions)
    return max(range(n_actions), key=lambda a: Q[s][a])

## 7교시 · SARSA와 Q-Learning 구현

`16:30 ~ 17:30` · `cliff_sarsa_qlearning.py`

- Gymnasium CliffWalking 환경에서 SARSA와 Q-Learning을 완성한다
- 두 알고리즘이 학습한 경로의 차이를 직접 확인한다


In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make("CliffWalking-v1")
n_states, n_actions = env.observation_space.n, env.action_space.n
alpha, gamma, epsilon = 0.1, 0.99, 0.1
rng = np.random.default_rng(0)

def eps_greedy(Q, s):
    if rng.random() < epsilon:
        return rng.integers(n_actions)
    return int(np.argmax(Q[s]))

def train(method, episodes=500):
    Q = np.zeros((n_states, n_actions))
    returns = []
    for _ in range(episodes):
        s, _ = env.reset()
        a = eps_greedy(Q, s)
        total, done = 0, False
        while not done:
            s_next, r, term, trunc, _ = env.step(a)
            done = term or trunc
            if method == "sarsa":
                a_next = eps_greedy(Q, s_next)
                target = r + gamma * Q[s_next][a_next] * (not done)
            else:  # q-learning
                target = r + gamma * Q[s_next].max() * (not done)
                a_next = eps_greedy(Q, s_next)
            Q[s][a] += alpha * (target - Q[s][a])
            s, a, total = s_next, a_next, total + r
        returns.append(total)
    return Q, returns

Q_sarsa, ret_s = train("sarsa")
Q_qlearn, ret_q = train("qlearning")
print(f"SARSA      마지막 100ep 평균 보상: {np.mean(ret_s[-100:]):.1f}")
print(f"Q-Learning 마지막 100ep 평균 보상: {np.mean(ret_q[-100:]):.1f}")

# greedy 경로 시각화: SARSA는 위쪽 안전 경로, Q-Learning은 절벽 옆 최단 경로
for name, Q in [("SARSA", Q_sarsa), ("Q-Learning", Q_qlearn)]:
    grid = np.full(48, '.', dtype=str)
    s, _ = env.reset()
    for _ in range(30):
        a = int(np.argmax(Q[s]))
        grid[s] = '*'
        s, r, term, trunc, _ = env.step(a)
        if term or trunc: break
    print(f"\n[{name} greedy 경로]")
    print('\n'.join(''.join(row) for row in grid.reshape(4, 12)))